# Introduction

This notebook processes and prepares the **[BrainMap](http://brainmap.org/)**  dataset for downstream analysis.

In [5]:
from tqdm import tqdm
import joblib
from nimare.io import convert_sleuth_to_dataset
import hashlib
from pathlib import Path
import warnings
import re


warnings.filterwarnings("ignore", category=FutureWarning)

DATA_PATH = Path('../data')
PLOTS_PATH = Path('../plots')
RESULTS_PATH = Path('../results')

# Comprehensive Data Acquisition

Because direct programmatic access to the metadata registry is restricted, we utilized **[Sleuth](http://brainmap.org/software.html#Sleuth)**, the official desktop application, to query and retrieve the empirical datasets. To ensure the utmost completeness of our dataset, we aimed to download all available data from the entire database using a rigorous, two-pass retrieval strategy:

1. **Primary Pass (By Cognitive Function):** We first performed systematic downloads categorizing the database by all available behavioral domains and cognitive functions.
2. **Secondary Pass (By Publication Year):** To cross-verify and capture any potential omissions, we conducted a second exhaustive round of downloads based on publication years.

This dual-approach strategy guaranteed maximum data coverage, resulting in near-perfect retrieval (4341/4341 papers, 22298/22302 experiments).

For each retrieved condition, Sleuth automatically generates a standardized triplet of files exported locally (**MNI coordinates**):
* **`.xml` file**: Contains the structured metadata regarding experimental conditions, contrast definitions, and subject cohorts.
* **`_locations.txt` file**: Contains the stereotaxic coordinates (in Talairach or MNI space) of the reported activation foci.
* **`_citations.txt` file**: Contains the bibliographic information and publication identifiers (e.g., PubMed IDs) for the included studies.

These exported file triplets serve as the raw input for our Python parsing pipeline, allowing us to map the precise spatial distribution of coordinates across the whole brain and specifically within our regions of interest.

Conditions are:
- 1985_2000
- 2000_2001
- 2001_2005
- 2005_2006
- 2006_2008
- 2008_2009
- 2009_2010
- 2010_2011
- 2011_2015
- 2015_2016
- 2016_2027
- activations_action
- activations_cognition_attention
- activations_cognition_language
- activations_cognition_memory
- activations_cognition_music
- activations_cognition_reasoning
- activations_cognition_socialcognition
- activations_cognition_somatic
- activations_cognition_spatial
- activations_cognition_temporal
- activations_emotion
- activations_interoception
- activations_perception
- deactivations


In [ ]:
raw_data_path = DATA_PATH / 'brainmap_data/1_raw_full'
exp_hash_map_text_dict = dict()
file_path_list = list(raw_data_path.glob("*_locations.txt"))
# we exported as MNI coordinates
reference_space = 'MNI'
for file_path in tqdm(file_path_list):
    line_list = file_path.read_text(errors="replace").splitlines()

    current_block = []
    has_coords = False

    for line_str in line_list:
        # skip empty line
        line_str = line_str.strip()
        if not line_str:
            continue

        # skip reference definition line
        if line_str.startswith("// Reference="):
            continue

        # encounter new line, save previous experiment info
        is_new_line = line_str.startswith("//") and has_coords
        if is_new_line:
            block_str = "\n".join(current_block)
            exp_hash = hashlib.md5(block_str.encode()).hexdigest()
            exp_hash_map_text_dict[exp_hash] = block_str
            current_block = []
            has_coords = False

        current_block.append(line_str)
        if not line_str.startswith("//"):
            has_coords = True

    # save the last experiment info
    if current_block and has_coords:
        block_str = "\n".join(current_block)
        exp_hash = hashlib.md5(block_str.encode()).hexdigest()
        exp_hash_map_text_dict[exp_hash] = block_str


collection_file_path = DATA_PATH / 'brainmap_data' / 'unique.txt'
with collection_file_path.open("w", newline="") as f:
    f.write(f"// Reference={reference_space}\n\n")
    for block_str in exp_hash_map_text_dict.values():
        f.write(block_str + "\n\n")

collection = convert_sleuth_to_dataset(str(collection_file_path), target="mni152_2mm")
joblib.dump(collection, DATA_PATH / 'brainmap_data' / 'brainmap.dataset.pkl')

In [ ]:

unique_paper_id_set = set()
file_path_list = list(raw_data_path.glob("*_citations.txt"))
for file_path in file_path_list:
    for line_str in file_path.read_text(errors="replace").splitlines():
        line_str = line_str.strip()
        if line_str.startswith("%2 "):
            brainmap_id = line_str.split("=", 1)[-1].strip()
            if brainmap_id:
                unique_paper_id_set.add(brainmap_id)

print(f"Total true unique papers : {len(unique_paper_id_set)}")
print(f"Total true unique experiments : {len(exp_hash_map_text_dict)}")

# Paradigm Classes Data Acquisition

Because direct programmatic access to the BrainMap metadata registry is restricted, we utilized **Sleuth**, the official desktop application provided by BrainMap, to perform targeted database queries and retrieve the required empirical datasets.

The data acquisition and export workflow consisted of the following steps:
1. **Query & Filtering:** We executed independent search queries within Sleuth, filtering the database by our target paradigm classes (i.e., *Affective Pictures* and *Affective Words*).
2. **Data Export:** For each verified paradigm category, the corresponding search results were exported locally. Sleuth automatically generates a standardized triplet of files for each session :
   - **`.xml` file**: Contains the structured metadata regarding experimental conditions, contrast definitions, and subject cohorts.
   - **`_locations.txt` file**: Contains the stereotaxic coordinates (in **MNI** space) of the reported activation foci.
   - **`_citations.txt` file**: Contains the bibliographic information and publication identifiers (e.g., PubMed IDs) for the included studies.

These exported files serve as the raw input for the subsequent Python parsing pipeline in this notebook, allowing us to map the precise spatial distribution of coordinates across the whole brain and specifically within the VMPFC.



In this case, we downloaded following paradigm classes:
- Induced Panic
- Sexual Arousal Gratification
- Trauma Recall
- Self-Reflection
- Face Monitor/Discrimination
- Emotional Body Language Perception
- Taste
- Hunger Satiety
- Emotion Induction
- Affective Pictures
- Figurative Language
- Reward
- Theory of Mind
- Competition Cooperation
- Affective Words
- Deception
- Gambling
- Delay Discounting
- Pain Monitor/Discrimination
- Classical Conditioning
- Episodic Recall
- Stroop Emotional
- Drug Challenge

In [6]:
exp_id_map_paradigm_list_dict = {}
raw_paradigm_path = DATA_PATH / 'brainmap_data/1_raw_sample_paradigm'
assert raw_paradigm_path.exists()

file_path_list = list(raw_paradigm_path.rglob('*_locations.txt'))
for file_path in tqdm(file_path_list):
    paradigm = file_path.name.replace('brainmap_', '').replace('_locations.txt', '')
    line_list = file_path.read_text(encoding="latin-1").splitlines()

    idx = 0
    while idx < len(line_list):
        line_str = line_list[idx].strip()
        if line_str.startswith("// Reference=") or line_str.startswith("// Subjects="):
            idx += 1;
            continue
        if not line_str.startswith("//"):
            idx += 1;
            continue

        header = line_str[3:]
        idx += 1
        # 将多行的 header 注释拼接在一起
        while idx < len(line_list) and line_list[idx].strip().startswith("//") and not (
                line_list[idx].strip().startswith("// Subjects=") or line_list[idx].strip().startswith(
                "// Reference=")):
            header += " " + line_list[idx].strip()[3:].strip()
            idx += 1

        matched = re.match(r"^(.+?,\s*\d{4}):\s+(.+)$", header)
        if matched:
            exp_id = f"{matched.group(1).strip()}-{matched.group(2).strip()}"
            exp_id_map_paradigm_list_dict.setdefault(exp_id, []).append(paradigm)

joblib.dump(exp_id_map_paradigm_list_dict, DATA_PATH / 'brainmap_data/exp_id_map_paradigm.dict.pkl')

100%|██████████| 23/23 [00:00<00:00, 103.42it/s]


['../data/brainmap_data/exp_id_map_paradigm.dict.pkl']